In [8]:
!pip install kagglehub


[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
from dotenv import load_dotenv
from pathlib import Path
import os
import shutil
import pandas as pd
import kagglehub

load_dotenv()

os.environ["KAGGLE_API_TOKEN"] = os.getenv("KAGGLE_API_TOKEN")

base_path_dataset = Path(os.getenv("BASE_PATH_DATASET"))
base_path_dataset.mkdir(parents=True, exist_ok=True)

kaggle_path = Path(kagglehub.dataset_download("ihelon/coffee-sales"))

c:\Users\carolina.bezerra\venv_3_10\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
kaggle_path

WindowsPath('C:/Users/carolina.bezerra/.cache/kagglehub/datasets/ihelon/coffee-sales/versions/21')

In [3]:
for file_path in kaggle_path.iterdir():
    if file_path.is_file():
        shutil.copy2(file_path, base_path_dataset / file_path.name)

csv_files = list(base_path_dataset.glob("*.csv"))


In [4]:
df = pd.read_csv(csv_files[0])
df.head()

,date,datetime,cash_type,card,money,coffee_name
0,2024-03-01,2024-03-01 10:15:50.520,card,ANON-0000-0000-0001,38.7,Latte
1,2024-03-01,2024-03-01 12:19:22.539,card,ANON-0000-0000-0002,38.7,Hot Chocolate
2,2024-03-01,2024-03-01 12:20:18.089,card,ANON-0000-0000-0002,38.7,Hot Chocolate
3,2024-03-01,2024-03-01 13:46:33.006,card,ANON-0000-0000-0003,28.9,Americano
4,2024-03-01,2024-03-01 13:48:14.626,card,ANON-0000-0000-0004,38.7,Latte


In [33]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3636 entries, 0 to 3635
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   date         3636 non-null   object 
 1   datetime     3636 non-null   object 
 2   cash_type    3636 non-null   object 
 3   card         3547 non-null   object 
 4   money        3636 non-null   float64
 5   coffee_name  3636 non-null   object 
dtypes: float64(1), object(5)
memory usage: 170.6+ KB


## Padronizar colunas

In [5]:
date_cols = ["date", "datetime"]
str_cols = ["cash_type", "card", "coffee_name"]

df[date_cols] = df[date_cols].apply(pd.to_datetime, errors="coerce")

df[str_cols] = (
    df[str_cols]
    .astype("string")
    .apply(lambda col: col.str.strip().str.lower())
)


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3636 entries, 0 to 3635
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   date         3636 non-null   datetime64[ns]
 1   datetime     3636 non-null   datetime64[ns]
 2   cash_type    3636 non-null   string        
 3   card         3547 non-null   string        
 4   money        3636 non-null   float64       
 5   coffee_name  3636 non-null   string        
dtypes: datetime64[ns](2), float64(1), string(3)
memory usage: 170.6 KB


## Checar qualidade

In [7]:
df.isna().sum()

date            0
datetime        0
cash_type       0
card           89
money           0
coffee_name     0
dtype: int64

In [8]:
df.groupby("cash_type")["card"].apply(lambda x: x.isna().sum())

cash_type
card     0
cash    89
Name: card, dtype: int64

In [ ]:
df[["cash_type", "card"]].value_counts(dropna=False) # verifica contagem de tipos em cada categoria 

cash_type  card               
card       anon-0000-0000-0012    129
           anon-0000-0000-0141    108
cash       <NA>                    89
card       anon-0000-0000-1163     77
           anon-0000-0000-0276     69
                                 ... 
           anon-0000-0000-0021      1
           anon-0000-0000-1303      1
           anon-0000-0000-1291      1
           anon-0000-0000-1292      1
           anon-0000-0000-1296      1
Name: count, Length: 1317, dtype: int64

In [11]:
invalid_cash_with_card = df[
    (df["cash_type"] == "cash") & (df["card"].notna())
]

invalid_card_without_card = df[
    (df["cash_type"] == "card") & (df["card"].isna())
]

print("Cash com card preenchido:", len(invalid_cash_with_card))
print("Card sem identificação de card:", len(invalid_card_without_card))

Cash com card preenchido: 0
Card sem identificação de card: 0


In [12]:
df["cash_type"].value_counts(dropna=False)

cash_type
card    3547
cash      89
Name: count, dtype: Int64

In [ ]:
date_mismatch = df[
    df["date"] != df["datetime"].dt.normalize()
]
len(date_mismatch) #check if there are any rows where the date and datetime columns do not match

0

In [ ]:
df.duplicated().sum() # check for duplicate rows in the DataFrame

np.int64(0)

In [ ]:
possible_duplicates = df.duplicated(
    subset=["datetime", "cash_type", "card", "money", "coffee_name"],
    keep=False
)

df[possible_duplicates].sort_values("datetime") #check for possible duplicates based on the specified subset of columns

,date,datetime,cash_type,card,money,coffee_name


## Checa valores de venda

In [17]:
df["money"].describe()

count    3636.000000
mean       31.746859
std         4.919926
min        18.120000
25%        27.920000
50%        32.820000
75%        35.760000
max        40.000000
Name: money, dtype: float64

In [18]:
df[df["money"] <= 0]

,date,datetime,cash_type,card,money,coffee_name


In [20]:
df.groupby("coffee_name")["money"].describe().sort_values("mean", ascending=False)

,count,mean,std,min,25%,50%,75%,max
coffee_name,,,,,,,,
hot chocolate,282.0,36.072553,1.527680,32.82,35.76,35.76,35.76,40.0
cappuccino,501.0,35.996287,1.906202,32.82,35.76,35.76,37.72,40.0
cocoa,243.0,35.712593,1.307759,32.82,35.76,35.76,35.76,40.0
latte,782.0,35.634655,1.936673,32.82,35.76,35.76,35.76,40.0
americano with milk,824.0,30.666408,1.940697,27.92,27.92,30.86,32.82,35.0
americano,578.0,26.059273,1.740508,23.02,25.96,25.96,25.96,30.0
cortado,292.0,25.804315,2.148458,23.02,23.02,25.96,27.92,30.0
espresso,134.0,21.002090,2.078350,18.12,18.12,21.06,23.02,25.0
